# Traffic Regulation Order (TRO) Extraction Pipeline (Part 1: PDF --> Structured CSV)

Traffic Regulation Orders (TROs) are the legal PDF documents UK local authorities publish
whenever they add, change, or revoke a parking / waiting / loading restriction on a street.
They're public, but they're locked inside scanned or lightly-formatted PDFs, there's no
structured schema behind them (this is exactly the gap the UK's new
[D-TRO digital order initiative](https://www.gov.uk/guidance/digital-traffic-regulation-orders-d-tro-service)
is trying to close).

This notebook walks through the pipeline step by step:

1. Render each PDF page as an image (TROs are usually image-heavy / inconsistently formatted,
   so a vision-capable LLM handles the layout far more robustly than text extraction).
2. Detect which pages belong to which numbered **Schedule** (e.g. "SCHEDULE III — Limited Waiting").
3. Extract every bylaw row (street, side of street, restriction length, raw text) within each
   Schedule as structured CSV rows.
4. Pull document-level metadata (title, effective date, penalties, enforcement authority).
5. Clean, forward-fill, and merge everything into one tidy CSV per order.

**Part 2** (a separate notebook) geocodes the resulting street descriptions into point/segment
geometries and compares them against councils that already publish official GIS layers.

## Sample data
**Recommended: [Transport Appeals TRO Library](https://tro.transportappeals.scot/authority_tro/?authority=North%20Lanarkshire%20Council)**
which is North Lanarkshire Council's own archive, 9 scanned orders in this exact format. Download
[NL001 - Airdrie Area (2018)](https://tro.transportappeals.scot/TRO/North%20Lanarkshire%20Council/20182f01-North-Lanarkshire-Council-Airdrie-Area-Traffic-Regulation-Consolidation-Order-2018_.pdf),
save it as `Inputs/NL001_airdrie_area_2018.pdf`, and you're ready to run this notebook as-is.


## 0. Setup

In [ ]:
# pip install pymupdf pillow openai python-dotenv pandas
import base64
import io
import os
import json
import re
import time
from pathlib import Path

import fitz  # PyMuPDF
import openai
import pandas as pd
from dotenv import load_dotenv
from PIL import Image
from IPython.display import Image as IPImage, display

load_dotenv()


In [ ]:
# --- Config: edit these for your PDF ---
PDF_PATH = "../Inputs/N001-20182f01-North-Lanarkshire-Council-Airdrie-Area-Traffic-Regulation-Consolidation-Order-2018_.pdf"
ORDER_ID = "NL001-airdrie-2018"
MODEL = "gpt-4o"
DPI = 200
SCHEDULE_SCAN_START_PAGE = 3     # first page likely to contain a "SCHEDULE" heading
METADATA_LAST_PAGE = 3           # page where penalties/enforcement info usually sits
REQUEST_DELAY_S = 0.5            # give the API a break between calls
MAX_RETRIES = 3

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

PROCESSED_DIR = Path("../Processed_Bylaws")
SCHEDULES_DIR = Path("../Schedules")
FINAL_DIR = Path("../Final")
for d in (PROCESSED_DIR, SCHEDULES_DIR, FINAL_DIR):
    d.mkdir(parents=True, exist_ok=True)

client = openai.OpenAI(api_key=OPENAI_API_KEY)  # reads OPENAI_API_KEY from the environment

FIELDS = ["Area", "Bylaw_Schedule", "Street", "Side_of_Street", "Length", "Raw_Text", "Raw_Bylaw_Text"]


## 1. Load the PDF and preview a page

Rendering pages to images (rather than extracting text) is what makes this robust against
scanned PDFs and the wildly inconsistent layouts different councils use.


In [ ]:
def page_to_png_base64(page, dpi=DPI):
    """Render a single PDF page to a base64-encoded PNG string."""
    pix = page.get_pixmap(dpi=dpi)
    img = Image.open(io.BytesIO(pix.tobytes()))
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode("utf-8")


doc = fitz.open(PDF_PATH)
print(f"Loaded '{PDF_PATH}' — {len(doc)} pages")


In [ ]:
# Sanity check: render and display the first page so we can see what we're working with
preview_b64 = page_to_png_base64(doc[0])
display(IPImage(data=base64.b64decode(preview_b64), width=500))


## 2. Detect Schedule sections

Each TRO groups its street-level restrictions under numbered Schedules (e.g. `SCHEDULE III`),
which is where the *type* of restriction (no waiting, limited waiting, loading ban, etc.) is
defined. We scan every page from `SCHEDULE_SCAN_START_PAGE` onward, asking the model to report
the Schedule heading (if any) visible on that page.


In [ ]:
SCHEDULE_DETECT_PROMPT = """
You are reading a page from a traffic bylaw.

Identify whether this page contains a genuine SCHEDULE heading — the title that marks the
start of a new schedule's table of street restrictions.

A genuine heading:
- appears on its own line, usually centered, near the top of the page
- is immediately followed by a restriction description (e.g. "NO WAITING AT ANY TIME") and/or
  a table of streets with columns like "Length(s) of Road" / "Side of Road"

This is NOT a genuine heading:
- a schedule number mentioned inside a sentence of running legal text, e.g. "...as specified
  in Schedule VI" or "...in accordance with Schedule XIV" — these are cross-references within
  the Articles of the Order, not page headings, and must be ignored

Return only the title in the format: SCHEDULE (Roman numeral), e.g., SCHEDULE III.
If no genuine schedule heading is present on the page, return: NONE.

Do not explain. Do not include extra text. Only return the exact string.
"""

page_titles = []  # list of (page_number, schedule_title)

for i in range(SCHEDULE_SCAN_START_PAGE, len(doc)):
    image_b64 = page_to_png_base64(doc[i])

    for attempt in range(MAX_RETRIES):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": "You are a helpful assistant."},
                    {"role": "user", "content": [
                        {"type": "text", "text": SCHEDULE_DETECT_PROMPT},
                        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
                    ]},
                ],
                temperature=0,
            )
            title = response.choices[0].message.content.strip().rstrip(".").upper()
            break
        except Exception as exc:
            print(f"  [retry {attempt + 1}/{MAX_RETRIES}] {exc}")
            time.sleep(2 * (attempt + 1))
    else:
        title = "NONE"

    print(f"Page {i + 1}: {title}")
    page_titles.append((i + 1, title))
    time.sleep(REQUEST_DELAY_S)


In [ ]:
# Save the raw page-by-page scan to file, and take a look at it
schedule_log_path = SCHEDULES_DIR / f"{ORDER_ID}_schedule_pages.txt"
with open(schedule_log_path, "w", encoding="utf-8") as f:
    for page, title in page_titles:
        f.write(f"Page {page}:{title}\n")

print(f"Saved to {schedule_log_path}\n")
page_titles


In [ ]:
# Collapse consecutive pages into (start_page, end_page, schedule_title) ranges
sections = []
current_start, current_title = None, None

for i, (page, title) in enumerate(page_titles):
    if title.startswith("SCHEDULE"):
        if current_start is not None:
            end_page = page_titles[i - 1][0]
            sections.append((current_start, end_page + 1, current_title))
        current_start, current_title = page, title

if current_start is not None:
    last_page = page_titles[-1][0]
    sections.append((current_start, last_page + 1, current_title))

print("Schedule ranges found:")
sections


## 3. Extract bylaw rows per Schedule section

For each Schedule's page range, ask the model to parse every bylaw row into a CSV line
matching a fixed set of fields. Keeping the field list and the parsing rules in the prompt
(rather than post-hoc regex) is what makes this robust across the very different layouts
different councils use for the same legal concept.


In [ ]:
BYLAW_EXTRACTION_PROMPT = f"""
You are given one or more images of pages from a bylaw PDF.
 
Goal: Parse all bylaws visible into a structured CSV format.
Only process a page if it has a parking bylaw table, underlined street names, or schedules.
Otherwise, output nothing for that page.
 
Output: Only a valid CSV string, no headers, no explanations, no markdown fences.
Fields (in this exact order): {FIELDS}. Use "" for any missing value.
 
RULES:
 
1. If the underlined title on the page has bylaws underneath starting with street names
   (not "From"), the underlined title is the Area, and each numbered sub-item under a
   street name becomes its own row with that street name (without the number).
   Example: "Liberty Road" with items (1) and (2) underneath becomes 2 rows, both with
   Street = "Liberty Road".
   Length is always the distance in metres as a bare number, e.g. "31 metres or thereby" becomes "31".
 
2. If a page has no bylaws but contains "ORDERS TO BE AMENDED OR REVOKED", parse each row as:
   Bylaw_Schedule = "I", Raw_Bylaw_Text = "SCHEDULE I NO STOPPING AT ANY TIME",
   Raw_Text = the row text (no commas), all other fields = "".
   Every row MUST still contain exactly 7 comma-separated values in the field order below,
   even when several are empty. Do not omit empty fields or shift columns. For example:
   "","I","","","","The Burgh of Example (Traffic Regulation) Order 1975 to be revoked in its entirety.","SCHEDULE I NO STOPPING AT ANY TIME"
 
3. Field definitions:
   - Area: ONLY the name of a sub-area from an underlined title grouping several streets
     together (e.g. an underlined "Bellshill" heading above a group of streets), or from a
     "Street, Area" format. Do NOT use the council or region name that appears in column
     headers like "Length(s) of Road in the Region of North Lanarkshire" - that is not an
     Area. If no genuine sub-area heading is present on the page, leave this "".
   - Bylaw_Schedule: the Roman numeral following "SCHEDULE" at the top of the page, if present.
   - Street: the street name.
   - Side_of_Street: "North" / "South" / "East" / "West", or a combination.
   - Length: distance in metres as a bare number.
   - Raw_Text: the full bylaw row text, with commas removed.
   - Raw_Bylaw_Text: the full bylaw text starting from the word "SCHEDULE" if present.
 
4. Never include commas inside a field's own text - they will break the CSV.
 
Keys MUST match {FIELDS} and stay in that order.
"""
 
all_rows = []  # raw CSV line strings, one per bylaw row, across every section
 
for start_page, end_page, title in sections:
    print(f"Extracting pages {start_page}-{end_page} ({title})")
 
    images = [page_to_png_base64(doc[p - 1]) for p in range(start_page, end_page)]
    content = [{"type": "text", "text": BYLAW_EXTRACTION_PROMPT}]
    content += [{"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}} for b64 in images]
 
    for attempt in range(MAX_RETRIES):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": "You are a helpful assistant."},
                    {"role": "user", "content": content},
                ],
                temperature=0,
            )
            csv_text = response.choices[0].message.content.strip()
            break
        except Exception as exc:
            print(f"  [retry {attempt + 1}/{MAX_RETRIES}] {exc}")
            time.sleep(2 * (attempt + 1))
    else:
        csv_text = ""
 
    rows = [line for line in csv_text.strip().splitlines() if line.strip()]
    print(f"  -> {len(rows)} rows")
    all_rows.extend(rows)
    time.sleep(REQUEST_DELAY_S)
 
raw_csv_path = PROCESSED_DIR / f"{ORDER_ID}.csv"
with open(raw_csv_path, "w", newline="", encoding="utf-8") as f:
    for row in all_rows:
        f.write(row + "\\n")
 
print(f"\\nRaw extraction saved to {raw_csv_path} ({len(all_rows)} rows)")

In [ ]:
# Quick look at what came back, before we clean anything
print(f"Total raw rows: {len(all_rows)}\n")
for row in all_rows[:10]:
    print(row)


## 4. Extract document-level metadata

Separately from the per-schedule bylaw rows, pull order-level fields (title, effective date,
penalties, enforcement authority) from the cover page and the page that usually lists
penalty/enforcement details.


In [ ]:
def build_metadata_prompt(order_id):
    return f"""
You are given a page of a bylaw PDF.
 
Extract the following fields as a JSON object. No explanations, no markdown, no extra text.
 
{{
  "TRO Title": "",
  "Area": "",
  "Effective Date": "",
  "Linked Order Ref": "{order_id}",
  "Penalties": "",
  "Enforcement Authority": "",
  "Penalty Charge Details": ""
}}
 
- "TRO Title": the title text at the top of the page.
- "Area": the geographic area this order covers, usually appearing in the title between the
  council name and "(Traffic Regulation)", e.g. "North Lanarkshire Council AIRDRIE AREA
  (Traffic Regulation) (Consolidation) Order 2018" -> "Airdrie". This is NOT the council or
  region name (e.g. not "North Lanarkshire") - it is the specific town/district the order covers.
- "Effective Date": the date following "shall come into operation on", converted to YYYY-MM-DD.
- "Penalties": Schedule Roman numerals listed after the word "Schedules" on the page, comma-separated.
- "Enforcement Authority": the job title of the penalty issuer (e.g. "Parking Attendant"), or "" if absent.
- "Penalty Charge Details": the name of the penalty charge (e.g. "Penalty Charge Notice").
 
If a field is missing, use "". Output ONLY valid JSON.
"""
 
 
def call_for_metadata(page_index):
    image_b64 = page_to_png_base64(doc[page_index])
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": [
                {"type": "text", "text": build_metadata_prompt(ORDER_ID)},
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
            ]},
        ],
        temperature=0,
    )
    raw = response.choices[0].message.content.strip()
    if raw.startswith("```"):
        raw = raw.strip("`")
        if raw.startswith("json"):
            raw = raw[4:]
        raw = raw.strip()
    return json.loads(raw)

In [ ]:
cover_metadata = call_for_metadata(0)
print("Cover page metadata:")
cover_metadata


In [ ]:
penalties_metadata = call_for_metadata(METADATA_LAST_PAGE - 1)
print("Penalties page metadata:")
penalties_metadata


## 5. Clean the raw CSV

Raw model output needs a pass of cleanup before it's analysis-ready: strip stray quotes/commas
that break CSV structure, and drop a header row if the model echoed one.


In [ ]:
cleaned_lines = []

for line in all_rows:
    if "," not in line:
        continue
    row = re.split(r',(?=(?:[^"]*"[^"]*")*[^"]*$)', line.strip())
    fields = [cell.strip().strip('"').replace(",", "") for cell in row]
    if len(fields) != len(FIELDS):
        print(f"  [warn] expected {len(FIELDS)} fields, got {len(fields)}: {fields}")
    cleaned_lines.append(",".join(fields))

# drop a header row if the model echoed one
cleaned_lines = [line for line in cleaned_lines if not line.startswith("Area,")]

processed_csv_path = PROCESSED_DIR / f"{ORDER_ID}_processed.csv"
with open(processed_csv_path, "w", encoding="utf-8") as f:
    f.write(",".join(FIELDS) + "\n")
    for line in cleaned_lines:
        f.write(line + "\n")

print(f"Cleaned CSV saved to {processed_csv_path} ({len(cleaned_lines)} rows)")


In [ ]:
# Check the cleaned data
df = pd.read_csv(processed_csv_path)
print(df.shape)
df.head(10)


## 6. Attach metadata and forward-fill

A Schedule heading, Area name, or restriction description typically applies to every row
underneath it until the next heading appears — so we forward-fill those columns rather than
leaving them blank.


In [ ]:
df["TRO Title"] = cover_metadata.get("TRO Title", "")
df["Effective Date"] = cover_metadata.get("Effective Date", "")
df["Penalties"] = penalties_metadata.get("Penalties", "")
df["Enforcement Authority"] = penalties_metadata.get("Enforcement Authority", "")
df["Penalty Charge Details"] = penalties_metadata.get("Penalty Charge Details", "")
 
df.head()

In [ ]:
order_area = cover_metadata.get("Area", "").strip() or "UNKNOWN"
invalid_area_values = {"", "UNKNOWN", "NORTH LANARKSHIRE"}  # known bad values the model sometimes leaks in
 
last_schedule, last_area, last_bylaw_text = "", "", ""
 
for idx, row in df.iterrows():
    if pd.notna(row["Raw_Bylaw_Text"]) and row["Raw_Bylaw_Text"] not in ("", "UNKNOWN"):
        last_bylaw_text = row["Raw_Bylaw_Text"]
    else:
        df.loc[idx, "Raw_Bylaw_Text"] = last_bylaw_text
 
    row_area = str(row["Area"]).strip() if pd.notna(row["Area"]) else ""
    if row_area.upper() not in invalid_area_values:
        last_area = row_area  # a genuine per-page sub-area (e.g. Bellshill-style grouping)
        df.loc[idx, "Area"] = row_area
    else:
        df.loc[idx, "Area"] = last_area or order_area  # fall back to the cover-page Area
 
    if pd.isna(row["Bylaw_Schedule"]) or row["Bylaw_Schedule"] in ("", "UNKNOWN"):
        df.loc[idx, "Bylaw_Schedule"] = last_schedule
    else:
        last_schedule = row["Bylaw_Schedule"]
 
# Verify: every row should now have a non-blank Schedule and Area
print(df[["Area", "Bylaw_Schedule"]].isna().sum())
df.sample(5)

In [ ]:
final_csv_path = FINAL_DIR / f"{ORDER_ID}_processed_all_fields.csv"
df.to_csv(final_csv_path, index=False)

print(f"Final structured CSV saved to {final_csv_path}")
print(f"{len(df)} rows, {df['Street'].nunique()} unique streets, schedules found: {sorted(df['Bylaw_Schedule'].dropna().unique())}")
df.head()
